In [2]:
import pandas as pd

In [3]:
file_path = "../data/raw/hospital_specialty_2024/hospital_sp113.csv"

In [4]:
df_hospital = pd.read_csv(file_path)

In [7]:
df_hospital = pd.read_csv(file_path)

df_hospital.shape

(162, 47)

In [8]:
df_hospital.head()

,鄉鎮市區碼,醫院家數,家庭醫學科,內科,外科,兒科,婦產科,骨科,神經科,神經外科,...,植牙科,中醫一般科,中醫內科,中醫外科,中醫眼科,中醫兒科,中醫婦科,中醫傷科,中醫針灸科,中醫痔科
0,101,5,4,4,4,4,4,4,4,3,...,0,1,0,0,0,0,0,0,0,0
1,102,6,6,6,5,4,6,5,5,4,...,0,0,0,0,0,0,0,0,0,0
2,109,1,1,1,1,1,1,1,1,1,...,0,1,0,0,0,0,0,0,0,0
3,110,3,1,1,2,3,3,2,1,2,...,0,1,0,0,0,0,0,0,0,0
4,111,4,3,3,2,3,3,3,3,2,...,0,2,0,0,0,0,0,0,0,0


In [9]:
hospital_tainan = df_hospital[
    df_hospital["鄉鎮市區碼"].between(501, 538)
].copy()

hospital_tainan.shape

(13, 47)

In [10]:
hospital_tainan[["鄉鎮市區碼", "醫院家數", "復健科"]]

,鄉鎮市區碼,醫院家數,復健科
32,501,4,3
33,504,1,1
34,505,1,1
35,506,2,1
36,507,1,0
37,509,1,1
38,527,2,0
39,529,1,0
40,531,5,3
41,532,3,2


In [12]:
lookup_path = "../data/raw/hospital_specialty_2024/欄位說明.csv"

df_lookup = pd.read_csv(lookup_path)

In [13]:
lookup_2024 = df_lookup[
    ["鄉鎮市區碼", "鄉鎮市區名稱(103年以後適用)"]
].copy()

lookup_2024.columns = [
    "鄉鎮市區碼",
    "district_full"
]

lookup_tainan = lookup_2024[
    lookup_2024["鄉鎮市區碼"].between(501, 538)
].copy()

In [14]:
lookup_tainan.shape

(37, 2)

In [15]:
# 以完整的 37 區 codebook 當左表
df_hospital_tainan = lookup_tainan.merge(
    hospital_tainan[["鄉鎮市區碼", "復健科"]],
    on="鄉鎮市區碼",
    how="left"
)

df_hospital_tainan.shape

(37, 3)

In [16]:
df_hospital_tainan["復健科"].isna().sum()

np.int64(24)

In [17]:
df_hospital_tainan["復健科"] = (
    df_hospital_tainan["復健科"]
    .fillna(0)
    .astype(int)
)

In [18]:
df_rehab_hospital = df_hospital_tainan[
    ["district_full", "復健科"]
].copy()

df_rehab_hospital.columns = [
    "district",
    "rehab_hospital_count"
]

df_rehab_hospital["district"] = (
    df_rehab_hospital["district"]
    .str.removeprefix("臺南市")
)

In [19]:
print(df_rehab_hospital.shape)
print(df_rehab_hospital.isna().sum())
print(df_rehab_hospital["district"].duplicated().sum())

(37, 2)
district                0
rehab_hospital_count    0
dtype: int64
0


In [20]:
output_path = "../data/processed/rehab_hospitals_tainan_2024_clean.csv"

df_rehab_hospital.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)